# 05 — Transformaciones

Crear columnas nuevas, modificar valores existentes, y trabajar con tipos especiales como strings y fechas.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df  = pd.read_csv(TRAIN, low_memory=False)
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)
air['host_since'] = pd.to_datetime(air['host_since'])
df['Order Date']  = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date']   = pd.to_datetime(df['Ship Date'],  format='%d/%m/%Y')


## assign() — añadir columnas de forma encadenable

`assign()` devuelve un DataFrame nuevo con las columnas añadidas. A diferencia de `df['col'] = ...`, se puede encadenar con otras operaciones.

In [ ]:
# assign puede referenciar columnas recién creadas en el mismo assign
df2 = df.assign(
    revenue     = df['Sales'],
    revenue_iva = lambda x: (x['Sales'] * 1.21).round(2),
    dias_envio  = lambda x: (x['Ship Date'] - x['Order Date']).dt.days,
)
print(df2[['Sales', 'revenue_iva', 'dias_envio']].head())


## apply() — función personalizada por eje

In [ ]:
# apply en columna — se aplica a cada elemento de la Serie
def categorizar_venta(monto):
    if monto < 50:    return 'baja'
    elif monto < 500: return 'media'
    else:             return 'alta'

df['categoria_venta'] = df['Sales'].apply(categorizar_venta)
print(df['categoria_venta'].value_counts())
print()

# La misma lógica con cut() es más eficiente — ver más abajo


In [ ]:
# apply en DataFrame — axis=1 aplica a cada fila
def resumen_fila(row):
    return f"{row['Customer Name']} compró {row['Sub-Category']} por ${row['Sales']:.0f}"

df['resumen'] = df.apply(resumen_fila, axis=1)
print(df['resumen'].head(3))

# Nota: apply con axis=1 es lento en datasets grandes
# Para operaciones simples, las operaciones vectorizadas son mucho más rápidas


## map() — transformación elemento a elemento

In [ ]:
# map() aplica un diccionario o función a cada elemento de una Serie
# A diferencia de apply(), map() es específico de Series

region_map = {'West': 'Oeste', 'East': 'Este', 'South': 'Sur', 'Central': 'Centro'}
df['Region_es'] = df['Region'].map(region_map)
# Los valores que no están en el diccionario quedan como NaN
print(df['Region_es'].value_counts())


## Accessor str — operaciones de string en Serie

In [ ]:
# El accessor .str expone todos los métodos de string de Python
# pero aplicados a toda la Serie de forma vectorizada

nombres = air['name'].dropna()

print(nombres.str.upper().head(3))
print(nombres.str.len().describe())               # longitud de cada string
print(nombres.str.contains('apartment', case=False).sum())  # cuántos tienen 'apartment'

# str.split devuelve listas — expand=True crea columnas separadas
partes = air['host_location'].str.split(',', expand=True)
print(partes.head(3))

# str.extract con regex — captura grupos
# extraer el año de un string como '2024-11-28'
air['host_year'] = air['host_since'].astype(str).str.extract(r'(\d{4})')
print(air['host_year'].value_counts().head(5))


## Accessor dt — operaciones de fecha en Serie

In [ ]:
# El accessor .dt expone atributos y métodos de datetime
print(df['Order Date'].dt.year.value_counts().sort_index())
print()
print(df['Order Date'].dt.month_name().value_counts())
print()

# Atributos más comunes
print(df['Order Date'].dt.dayofweek.head())   # 0=lunes, 6=domingo
print(df['Order Date'].dt.quarter.head())     # trimestre
print(df['Order Date'].dt.is_month_end.head()) # ¿es el último día del mes?

# to_period convierte a Period (más flexible para agrupar por mes/año)
df['mes_periodo'] = df['Order Date'].dt.to_period('M')
print(df['mes_periodo'].head())


## cut() y qcut() — crear categorías desde valores numéricos

In [ ]:
# cut — bins con límites fijos
df['rango_precio'] = pd.cut(
    df['Sales'],
    bins=[0, 50, 200, 500, float('inf')],
    labels=['<50', '50-200', '200-500', '>500'],
    right=True   # el límite derecho de cada intervalo está incluido
)
print(df['rango_precio'].value_counts().sort_index())
print()

# qcut — bins por cuantiles (mismo número de elementos en cada bin)
df['cuartil_precio'] = pd.qcut(df['Sales'], q=4,
                                labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df['cuartil_precio'].value_counts())


---
## Resumen

| Operación | Sintaxis |
|-----------|----------|
| Añadir columna (encadenable) | `df.assign(nueva=lambda x: ...)` |
| Función personalizada por elemento | `df['col'].apply(func)` |
| Función por fila | `df.apply(func, axis=1)` |
| Mapear valores | `df['col'].map({'a': 1, 'b': 2})` |
| Operaciones de string | `df['col'].str.lower()`, `.str.contains()` |
| Operaciones de fecha | `df['col'].dt.year`, `.dt.month` |
| Binning fijo | `pd.cut(serie, bins=[...])` |
| Binning por cuantiles | `pd.qcut(serie, q=4)` |
